# KLDM DiffCSP++ Lattice-Mask Compatibility Tests

This notebook is a **pre-implementation compatibility harness** for a KLDM lattice-mask variant that would replace the native KLDM lattice state with a DiffCSP++-style masked `k`-space lattice variable.

It does **not** ask whether training improves `match_rate` or `rmse` yet.
It asks the earlier question:

> Can the current KLDM data pipeline, batching, score network, and sampler safely accept DiffCSP++ lattice masking without creating inconsistent `(F, L)` pairs or broken tensors?

The notebook keeps the implementation temporary and notebook-local:
- it uses the current KLDM dataset/model code as-is,
- it imports DiffCSP++ `CrystalFamily` directly,
- it imitates masked `k`-space noising/sampling in helper functions,
- it does **not** change network code or training code.


## Test Map

### Test 0: Batch Field Compatibility
Verify the current KLDM batch already exposes the fields required by the masked-lattice path.

### Test 1: `L -> de_so3(L) -> k -> L` Roundtrip
Check that DiffCSP++ lattice operators are numerically stable on current KLDM cells.

### Test 2: Rotation-Gauge Consistency
Check that replacing `L` by `de_so3(L)` preserves periodic geometry for the same fractional coordinates.

### Test 3: Projection Error
Measure how much `proj_k_to_spacegroup` changes the current KLDM cells.

### Test 4: Projected-Lattice Geometry Distortion
Measure how much `(F, v2m(proj(k)))` changes the structure geometry.

### Test 5: Structure / Space-Group Compatibility
Use `StructureMatcher` and `SpacegroupAnalyzer` to see whether projected lattices remain compatible with the original structure and label family.

### Test 6: Masked VP Forward Noising Dry Run
Imitate masked `k`-space forward noising with the current KLDM VP lattice diffusion.

### Test 7: Network Dry Forward
Feed masked `k_t` directly into the unchanged `CSPVNet` and check shapes / finiteness.

### Test 8: One-Step Masked Reverse Dry Run
Imitate one masked lattice reverse step and verify constraints / SPD decoding / output conversion.

### Summary
Aggregate the compatibility results into a simple go/no-go table.


In [1]:
from __future__ import annotations

import json
import math
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader, Subset
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_ROOT = ROOT / 'src'
DIFFCSPPP_ROOT = ROOT / 'src' / 'diffCSPpp' / 'DiffCSP-PP-main'
for path in (SRC_ROOT, DIFFCSPPP_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from diffcsp.pl_modules.lattice.crystal_family import CrystalFamily
from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.utils.model_loader import build_model, load_checkpoint
from kldmPlus.utils.time import make_times

try:
    from pymatgen.analysis.structure_matcher import StructureMatcher
    from pymatgen.core import Lattice, Structure
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
except ImportError:  # pragma: no cover
    StructureMatcher = Lattice = Structure = SpacegroupAnalyzer = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

torch.set_printoptions(precision=6, sci_mode=False)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.get_default_dtype()
print(f'ROOT={ROOT}')
print(f'DEVICE={DEVICE}')


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
ROOT=/workspace
DEVICE=cpu


In [2]:
CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_lattice_eps_pc.yaml'
CHECKPOINT_PATH = Path('/workspace/artifacts/HPC/checkpoints/epoch_8900.pt')
CHECKPOINT_SUFFIXES = {'.pt', '.pth', '.ckpt'}
DOWNLOAD_DATA = False
SPLIT = 'test'
SUBSET_SIZE = 8
BATCH_SIZE = 4
T_PROBE = 0.5
DT_REVERSE = 1.0 / 1000.0
SYMPREC_LIST = [1e-3, 1e-2, 1e-1]
ANGLE_TOL = 5.0
MATCHER_STOL = 0.5
MATCHER_ANGLE_TOL = 10.0
MATCHER_LTOL = 0.3
ROUNDTRIP_TOL = 1e-5
DIST_TOL = 1e-5
CONSTRAINED_TOL = 1e-6

config = yaml.safe_load(CONFIG_PATH.read_text())
checkpoint_suffix = CHECKPOINT_PATH.suffix.lower()
LOAD_CHECKPOINT = CHECKPOINT_PATH.exists() and checkpoint_suffix in CHECKPOINT_SUFFIXES
print(f'CONFIG_PATH={CONFIG_PATH}')
print(f'CHECKPOINT_PATH={CHECKPOINT_PATH} suffix={checkpoint_suffix} load_checkpoint={LOAD_CHECKPOINT}')
if CHECKPOINT_PATH.exists() and not LOAD_CHECKPOINT:
    print('Skipping checkpoint load: file does not look like a Torch checkpoint (.pt/.pth/.ckpt).')
print(f'split={SPLIT} subset_size={SUBSET_SIZE} batch_size={BATCH_SIZE} t_probe={T_PROBE}')


CONFIG_PATH=/workspace/configs/kldm_plus/mp_20/mp20_lattice_eps_pc.yaml
CHECKPOINT_PATH=/workspace/artifacts/HPC/checkpoints/epoch_8900.pt suffix=.pt load_checkpoint=True
split=test subset_size=8 batch_size=4 t_probe=0.5


In [3]:
dataset_cfg = dict(config['dataset'])
model_cfg = dict(config['model'])

task = CSPTask(
    dataset_name=str(dataset_cfg['name']),
    lattice_parameterization=str(model_cfg['lattice_parameterization']),
    lattice_representation=str(dataset_cfg.get('lattice_representation', 'kldm')),
)
root = resolve_data_root(dataset_cfg.get('root'))
dataset_full = task.fit_dataset(root=root, split=SPLIT, download=DOWNLOAD_DATA)
subset = Subset(dataset_full, list(range(min(SUBSET_SIZE, len(dataset_full)))))
loader = DataLoader(
    subset,
    batch_size=min(BATCH_SIZE, len(subset)),
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=dataset_full.collate_fn,
)
lattice_transform = task.make_lattice_transform(root=root, download=DOWNLOAD_DATA)

batch = next(iter(loader)).to(DEVICE)
model = build_model(config=config, device=DEVICE)
checkpoint_loaded = False
checkpoint_load_error = None
if LOAD_CHECKPOINT:
    try:
        load_checkpoint(
            checkpoint_path=CHECKPOINT_PATH,
            model=model,
            device=DEVICE,
            prefer_ema_weights=True,
        )
        checkpoint_loaded = True
    except Exception as exc:
        checkpoint_load_error = repr(exc)
        print(f'Checkpoint load failed, continuing with uninitialized model: {checkpoint_load_error}')
model.eval()
crystal_family = CrystalFamily().to(DEVICE)

print(f'batch.num_graphs={int(batch.num_graphs)} num_nodes={int(batch.pos.shape[0])}')
print(f'batch.l.shape={tuple(batch.l.shape)} batch.pos.shape={tuple(batch.pos.shape)} batch.atomic_numbers.shape={tuple(batch.atomic_numbers.shape)}')
print(f'lattice_representation={task.lattice_representation} lattice_parameterization={model.lattice_parameterization}')
print(f'checkpoint_loaded={checkpoint_loaded}')


dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
batch.num_graphs=4 num_nodes=21
batch.l.shape=(4, 6) batch.pos.shape=(21, 3) batch.atomic_numbers.shape=(21,)
lattice_representation=kldm lattice_parameterization=eps
checkpoint_loaded=True


## Helpers

The helpers below keep everything notebook-local. They decode KLDM lattice features into cell matrices, run DiffCSP++ `k`-space operators, build temporary masked forward/reverse lattice steps, and compute geometry/matching diagnostics.


In [4]:
FAMILY_LABELS = {
    1: 'triclinic',
    2: 'monoclinic',
    3: 'orthorhombic',
    4: 'tetragonal',
    5: 'hexagonal_or_trigonal',
    6: 'cubic',
}


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return float('nan')


def space_group_tensor(batch) -> torch.Tensor:
    for name in ('space_group', 'spacegroup'):
        if hasattr(batch, name):
            sg = torch.as_tensor(getattr(batch, name), device=batch.pos.device, dtype=torch.long).reshape(-1)
            return sg
    raise AttributeError('Batch is missing space_group / spacegroup.')


def family_from_sg(sg: int) -> str:
    if 195 <= sg <= 230:
        return 'cubic'
    if 143 <= sg <= 194:
        return 'hexagonal_or_trigonal'
    if 75 <= sg <= 142:
        return 'tetragonal'
    if 16 <= sg <= 74:
        return 'orthorhombic'
    if 3 <= sg <= 15:
        return 'monoclinic'
    return 'triclinic'


def graph_masks(batch):
    return [batch.batch == g for g in range(int(batch.num_graphs))]


def lengths_angles_to_matrix(lengths: torch.Tensor, angles: torch.Tensor) -> torch.Tensor:
    a, b, c = lengths.unbind(dim=-1)
    alpha, beta, gamma = angles.unbind(dim=-1)
    cos_alpha = torch.cos(alpha)
    cos_beta = torch.cos(beta)
    cos_gamma = torch.cos(gamma)
    sin_gamma = torch.sin(gamma).clamp_min(1e-8)
    row0 = torch.stack([a, torch.zeros_like(a), torch.zeros_like(a)], dim=-1)
    row1 = torch.stack([b * cos_gamma, b * sin_gamma, torch.zeros_like(b)], dim=-1)
    cx = c * cos_beta
    cy = c * (cos_alpha - cos_beta * cos_gamma) / sin_gamma
    cz_sq = (c.square() - cx.square() - cy.square()).clamp_min(1e-12)
    row2 = torch.stack([cx, cy, torch.sqrt(cz_sq)], dim=-1)
    return torch.stack([row0, row1, row2], dim=-2)


def lattice6_to_matrix(l: torch.Tensor, num_atoms: torch.Tensor | int, lattice_transform) -> torch.Tensor:
    if hasattr(lattice_transform, 'invert_to_matrix'):
        return lattice_transform.invert_to_matrix(l=l, num_atoms=num_atoms).reshape(*l.shape[:-1], 3, 3)
    lengths, angles = lattice_transform.invert_to_lengths_angles(l=l, num_atoms=num_atoms)
    return lengths_angles_to_matrix(lengths, angles)


def source_cells_from_batch(batch, lattice_transform) -> torch.Tensor:
    if hasattr(batch, 'cell'):
        cell = torch.as_tensor(batch.cell, device=batch.pos.device, dtype=batch.pos.dtype)
        if cell.ndim == 4 and cell.shape[1] == 1:
            return cell[:, 0]
        if cell.ndim == 3:
            return cell
    return lattice6_to_matrix(batch.l, batch.num_atoms, lattice_transform)


def pairwise_pbc_distances(frac: torch.Tensor, cell: torch.Tensor) -> torch.Tensor:
    delta = frac[:, None, :] - frac[None, :, :]
    delta = delta - torch.round(delta)
    cart = torch.einsum('...i,ij->...j', delta, cell)
    return torch.linalg.norm(cart, dim=-1)


def offdiag_values(matrix: torch.Tensor) -> torch.Tensor:
    n = matrix.shape[0]
    mask = ~torch.eye(n, dtype=torch.bool, device=matrix.device)
    return matrix[mask]


def mean_abs_relative_difference(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> float:
    denom = a.abs().clamp_min(eps)
    return float(torch.mean(torch.abs(a - b) / denom).detach().item())


def build_structure(cell: torch.Tensor, frac: torch.Tensor, atomic_numbers: torch.Tensor):
    if None in (Lattice, Structure):
        return None
    lattice = Lattice(cell.detach().cpu().numpy())
    species = [int(z) for z in atomic_numbers.detach().cpu().tolist()]
    coords = frac.detach().cpu().numpy()
    return Structure(lattice, species, coords, coords_are_cartesian=False)


def matcher_fit_and_rms(source, target):
    if source is None or target is None or StructureMatcher is None:
        return None, None
    matcher = StructureMatcher(stol=MATCHER_STOL, angle_tol=MATCHER_ANGLE_TOL, ltol=MATCHER_LTOL)
    fit = bool(matcher.fit(source, target))
    try:
        rms = matcher.get_rms_dist(source, target)
        if rms is None:
            rms_value = None
        elif isinstance(rms, (tuple, list)) and len(rms) > 0:
            first = rms[0]
            if isinstance(first, (tuple, list)) and len(first) > 0:
                rms_value = safe_float(first[0])
            else:
                rms_value = safe_float(first)
        else:
            rms_value = safe_float(rms)
    except Exception:
        rms_value = None
    return fit, rms_value


def detect_sg(structure, symprec=1e-2, angle_tolerance=ANGLE_TOL):
    if structure is None or SpacegroupAnalyzer is None:
        return None
    try:
        return int(SpacegroupAnalyzer(structure, symprec=symprec, angle_tolerance=angle_tolerance).get_space_group_number())
    except Exception:
        return None


def masks_and_biases(sg: torch.Tensor):
    return crystal_family.masks[sg], crystal_family.biass[sg]


def masked_k_forward_sample(k0: torch.Tensor, sg: torch.Tensor, t_lattice: torch.Tensor, diffusion_l):
    if diffusion_l.parameterization != 'eps':
        raise ValueError('Masked-k dry runs currently assume eps parameterization.')
    mask, bias = masks_and_biases(sg)
    z0 = mask * (k0 - bias)
    eps = mask * torch.randn_like(z0)
    alpha = diffusion_l._match_dims(diffusion_l.alpha(t_lattice), z0)
    sigma = diffusion_l._match_dims(diffusion_l.sigma(t_lattice), z0)
    z_t = alpha * z0 + sigma * eps
    k_t = bias + z_t
    return {
        'mask': mask,
        'bias': bias,
        'z0': z0,
        'eps': eps,
        'z_t': z_t,
        'k_t': k_t,
        'target_k': eps,
        'alpha': alpha,
        'sigma': sigma,
    }


def masked_k_reverse_step(k_t: torch.Tensor, pred_k: torch.Tensor, sg: torch.Tensor, t_lattice: torch.Tensor, dt: float, diffusion_l):
    mask, bias = masks_and_biases(sg)
    z_t = mask * (k_t - bias)
    pred_free = mask * pred_k
    z_prev = diffusion_l.reverse_step(
        t=t_lattice,
        x_t=z_t,
        pred=pred_free,
        dt=dt,
        num_atoms=None,
    )
    z_prev = mask * z_prev
    return {
        'mask': mask,
        'bias': bias,
        'z_prev': z_prev,
        'k_prev': bias + z_prev,
    }


def induced_cart_shift(frac: torch.Tensor, cell_before: torch.Tensor, cell_after: torch.Tensor):
    delta = cell_after - cell_before
    cart = frac @ delta
    norms = torch.linalg.norm(cart, dim=-1)
    rms = torch.sqrt(torch.mean(norms.square())) if norms.numel() else torch.tensor(0.0, device=frac.device)
    max_abs = torch.max(norms) if norms.numel() else torch.tensor(0.0, device=frac.device)
    return rms, max_abs


def standardize_structure_variants(structure):
    variants = [('raw_gt', structure)]
    if structure is None or SpacegroupAnalyzer is None:
        return variants
    try:
        sga = SpacegroupAnalyzer(structure, symprec=1e-2, angle_tolerance=ANGLE_TOL)
        variants.append(('refined_standardized', sga.get_refined_structure()))
        variants.append(('conventional_standardized', sga.get_conventional_standard_structure()))
    except Exception:
        pass
    return variants


def display_sorted_frame(rows, sort_cols):
    df = pd.DataFrame(rows)
    if len(df) and sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    display(df)
    return df



### Test 0: Batch Field Compatibility

In [5]:
rows_t0 = []
required_fields = ['pos', 'atomic_numbers', 'l', 'edge_node_index', 'batch']
for field in required_fields:
    exists = hasattr(batch, field)
    shape = tuple(getattr(batch, field).shape) if exists and hasattr(getattr(batch, field), 'shape') else None
    rows_t0.append({
        'field': field,
        'exists': bool(exists),
        'shape': shape,
        'PASS': bool(exists),
    })

sg = None
sg_error = None
try:
    sg = space_group_tensor(batch)
except Exception as exc:
    sg_error = repr(exc)
rows_t0.append({
    'field': 'space_group',
    'exists': sg is not None,
    'shape': tuple(sg.shape) if sg is not None else None,
    'PASS': bool(sg is not None and sg.numel() == int(batch.num_graphs)),
    'notes': sg_error,
})

rows_t0.append({
    'field': 'cell_or_decodable_lattice',
    'exists': True,
    'shape': tuple(source_cells_from_batch(batch, lattice_transform).shape),
    'PASS': True,
})

df_t0 = display_sorted_frame(rows_t0, ['field'])


,field,exists,shape,PASS,notes
0,atomic_numbers,True,"(21,)",True,NaN
1,batch,True,"(21,)",True,NaN
2,cell_or_decodable_lattice,True,"(4, 3, 3)",True,NaN
3,edge_node_index,True,"(2, 112)",True,NaN
4,l,True,"(4, 6)",True,NaN
5,pos,True,"(21, 3)",True,NaN
6,space_group,True,"(4,)",True,NaN


### Test 1: `L -> de_so3(L) -> k -> L` Roundtrip
### Test 2: Rotation-Gauge Consistency

In [6]:
source_cells = source_cells_from_batch(batch, lattice_transform)
sg = space_group_tensor(batch)
rows_t12 = []
for g, node_mask in enumerate(graph_masks(batch)):
    cell = source_cells[g].reshape(3, 3)
    frac = batch.pos[node_mask].reshape(-1, 3)
    l_sym = crystal_family.de_so3(cell[None])[0]
    k = crystal_family.m2v(l_sym[None])[0]
    l_rec = crystal_family.v2m(k[None])[0]
    rel_roundtrip = torch.linalg.norm((l_sym - l_rec).reshape(-1)) / torch.linalg.norm(l_sym.reshape(-1)).clamp_min(1e-12)
    d_orig = offdiag_values(pairwise_pbc_distances(frac, cell))
    d_sym = offdiag_values(pairwise_pbc_distances(frac, l_sym))
    rows_t12.append({
        'graph': g,
        'space_group': int(sg[g].item()),
        'family': family_from_sg(int(sg[g].item())),
        'det_source': safe_float(torch.det(cell)),
        'det_sym': safe_float(torch.det(l_sym)),
        'roundtrip_rel_error': safe_float(rel_roundtrip),
        'sym_distance_rmse': safe_float(torch.sqrt(torch.mean((d_orig - d_sym).square()))),
        'sym_distance_max_abs': safe_float(torch.max(torch.abs(d_orig - d_sym))),
        'PASS_roundtrip': bool(rel_roundtrip <= ROUNDTRIP_TOL),
        'PASS_rotation_gauge': bool(torch.max(torch.abs(d_orig - d_sym)) <= DIST_TOL),
    })

df_t12 = display_sorted_frame(rows_t12, ['graph'])


,graph,space_group,family,det_source,det_sym,roundtrip_rel_error,sym_distance_rmse,sym_distance_max_abs,PASS_roundtrip,PASS_rotation_gauge
0,0,194,hexagonal_or_trigonal,272.783752,272.783722,1.352465e-07,7.804080e-08,2.384186e-07,True,True
1,1,160,hexagonal_or_trigonal,71.949242,71.949280,3.234460e-07,4.013183e-07,7.152557e-07,True,True
2,2,65,orthorhombic,22.426287,22.426285,1.254414e-07,0.000000e+00,0.000000e+00,True,True
3,3,8,monoclinic,142.231735,142.231659,1.280317e-07,4.352908e-07,7.152557e-07,True,True


### Test 3: Raw / Standardized GT k Identity and Projection Distortion

This section separates three possible failure modes:

1. `raw_gt` fails: the hard mask is already incompatible with the current KLDM cell/frame.
2. `raw_gt` fails but standardized passes: the lattice mask itself may be fine, and structure standardization is the real issue.
3. standardized still fails: the family mask/projection is not geometry-preserving enough for that sample.



In [7]:
rows_t345 = []
source_cells = source_cells_from_batch(batch, lattice_transform)
sg = space_group_tensor(batch)
for g, node_mask in enumerate(graph_masks(batch)):
    raw_cell = source_cells[g].reshape(3, 3)
    frac = batch.pos[node_mask].reshape(-1, 3)
    atomic_numbers = batch.atomic_numbers[node_mask]
    s_raw = build_structure(raw_cell, frac, atomic_numbers)
    raw_num_sites = int(frac.shape[0])

    for cell_source, structure_variant in standardize_structure_variants(s_raw):
        if structure_variant is None:
            continue
        cell_variant = torch.as_tensor(structure_variant.lattice.matrix, device=raw_cell.device, dtype=raw_cell.dtype)
        frac_variant = torch.as_tensor(structure_variant.frac_coords, device=frac.device, dtype=frac.dtype)
        variant_atomic_numbers = torch.as_tensor([int(site.specie.Z) for site in structure_variant], device=atomic_numbers.device, dtype=atomic_numbers.dtype)
        num_sites_variant = int(len(structure_variant))

        l_sym = crystal_family.de_so3(cell_variant[None])[0]
        k = crystal_family.m2v(l_sym[None])[0]
        k_proj = crystal_family.proj_k_to_spacegroup(k[None], sg[g:g+1])[0]
        l_proj = crystal_family.v2m(k_proj[None])[0]

        d_sym = offdiag_values(pairwise_pbc_distances(frac_variant, l_sym))
        d_proj = offdiag_values(pairwise_pbc_distances(frac_variant, l_proj))
        distortion = mean_abs_relative_difference(d_sym, d_proj)
        distortion_max = safe_float(torch.max(torch.abs(d_sym - d_proj)))
        shift_rms, shift_max = induced_cart_shift(frac_variant, l_sym, l_proj)

        s_sym = build_structure(l_sym, frac_variant, variant_atomic_numbers)
        s_proj = build_structure(l_proj, frac_variant, variant_atomic_numbers)
        match_fit, match_rms = matcher_fit_and_rms(s_sym, s_proj)

        detected_sym = detect_sg(s_sym, symprec=1e-2)
        detected_proj = detect_sg(s_proj, symprec=1e-2)
        same_family_sym = (detected_sym is not None and family_from_sg(detected_sym) == family_from_sg(int(sg[g].item())))
        same_family_proj = (detected_proj is not None and family_from_sg(detected_proj) == family_from_sg(int(sg[g].item())))

        rows_t345.append({
            'graph': g,
            'cell_source': cell_source,
            'space_group': int(sg[g].item()),
            'family': family_from_sg(int(sg[g].item())),
            'num_sites_raw_gt': raw_num_sites,
            'num_sites_variant': num_sites_variant,
            'atom_count_delta_under_standardization': int(num_sites_variant - raw_num_sites),
            'projection_err_mean_abs': safe_float(torch.mean(torch.abs(k - k_proj))),
            'projection_err_max_abs': safe_float(torch.max(torch.abs(k - k_proj))),
            'induced_cart_shift_rms': safe_float(shift_rms),
            'induced_cart_shift_max': safe_float(shift_max),
            'distance_distortion_mean_rel': distortion,
            'distance_distortion_max_abs': distortion_max,
            'structure_match_orig_vs_proj': match_fit,
            'structure_rms_orig_vs_proj': match_rms,
            'detected_sg_sym': detected_sym,
            'detected_sg_proj': detected_proj,
            'same_family_as_label_sym': same_family_sym,
            'same_family_as_label_proj': same_family_proj,
            'PASS_projection_smallish': bool(torch.mean(torch.abs(k - k_proj)) < 0.25),
            'PASS_distortion_smallish': bool(distortion < 0.10),
            'PASS_induced_shift_smallish': bool(safe_float(shift_rms) < 0.25),
            'PASS_structure_match': bool(match_fit) if match_fit is not None else False,
        })

df_t345 = display_sorted_frame(rows_t345, ['graph', 'cell_source'])

df_t345_family = (
    df_t345.groupby(['family', 'cell_source'], dropna=False)
    .agg(
        num_graphs=('graph', 'count'),
        projection_pass_rate=('PASS_projection_smallish', 'mean'),
        distortion_pass_rate=('PASS_distortion_smallish', 'mean'),
        induced_shift_pass_rate=('PASS_induced_shift_smallish', 'mean'),
        structure_match_rate=('PASS_structure_match', 'mean'),
        mean_projection_err=('projection_err_mean_abs', 'mean'),
        mean_induced_cart_shift_rms=('induced_cart_shift_rms', 'mean'),
        mean_atom_count_delta=('atom_count_delta_under_standardization', 'mean'),
    )
    .reset_index()
)
display(df_t345_family)



,graph,space_group,family,projection_err_mean_abs,projection_err_max_abs,distance_distortion_mean_rel,distance_distortion_max_abs,structure_match_orig_vs_proj,structure_rms_orig_vs_proj,detected_sg_sym,detected_sg_proj,same_family_as_label_sym,same_family_as_label_proj,PASS_projection_smallish,PASS_distortion_smallish
0,0,194,hexagonal_or_trigonal,5.640518e-08,2.490241e-07,1.370422e-07,9.536743e-07,True,8.209840e-17,194,194,True,True,True,True
1,1,160,hexagonal_or_trigonal,1.823073e-01,7.508783e-01,2.155186e-01,8.269686e-01,False,NaN,160,8,True,False,True,False
2,2,65,orthorhombic,5.378701e-02,3.227220e-01,7.950682e-02,1.947832e-01,False,NaN,65,123,True,False,True,True
3,3,8,monoclinic,1.861035e-02,9.707060e-02,4.320458e-02,2.728672e-01,True,4.805224e-16,8,8,True,True,True,True


### Test 5b: Space-Group Agreement Across `symprec` Sweep

In [8]:
rows_t5b = []
source_cells = source_cells_from_batch(batch, lattice_transform)
sg = space_group_tensor(batch)
for g, node_mask in enumerate(graph_masks(batch)):
    frac = batch.pos[node_mask].reshape(-1, 3)
    atomic_numbers = batch.atomic_numbers[node_mask]
    cell = source_cells[g].reshape(3, 3)
    s_orig = build_structure(cell, frac, atomic_numbers)
    if s_orig is None:
        continue
    for symprec in SYMPREC_LIST:
        detected = detect_sg(s_orig, symprec=symprec)
        rows_t5b.append({
            'graph': g,
            'symprec': symprec,
            'label_sg': int(sg[g].item()),
            'detected_sg': detected,
            'label_family': family_from_sg(int(sg[g].item())),
            'detected_family': family_from_sg(detected) if detected is not None else None,
            'same_exact_sg': bool(detected == int(sg[g].item())) if detected is not None else False,
            'same_family': bool(detected is not None and family_from_sg(detected) == family_from_sg(int(sg[g].item()))),
        })

df_t5b = display_sorted_frame(rows_t5b, ['graph', 'symprec'])


,graph,symprec,label_sg,detected_sg,label_family,detected_family,same_exact_sg,same_family
0,0,0.001,194,194,hexagonal_or_trigonal,hexagonal_or_trigonal,True,True
1,0,0.010,194,194,hexagonal_or_trigonal,hexagonal_or_trigonal,True,True
2,0,0.100,194,194,hexagonal_or_trigonal,hexagonal_or_trigonal,True,True
3,1,0.001,160,160,hexagonal_or_trigonal,hexagonal_or_trigonal,True,True
4,1,0.010,160,160,hexagonal_or_trigonal,hexagonal_or_trigonal,True,True
5,1,0.100,160,160,hexagonal_or_trigonal,hexagonal_or_trigonal,True,True
6,2,0.001,65,65,orthorhombic,orthorhombic,True,True
7,2,0.010,65,65,orthorhombic,orthorhombic,True,True
8,2,0.100,65,65,orthorhombic,orthorhombic,True,True
9,3,0.001,8,8,monoclinic,monoclinic,True,True


### Test 6: Masked VP Forward Noising Dry Run

In [9]:
if model.diffusion_l.parameterization != 'eps':
    raise ValueError('This notebook currently assumes an eps lattice parameterization config.')

source_cells = source_cells_from_batch(batch, lattice_transform)
sg = space_group_tensor(batch)
l_sym = crystal_family.de_so3(source_cells)
k0 = crystal_family.m2v(l_sym)
times = make_times(batch, torch.full((int(batch.num_graphs),), float(T_PROBE), device=batch.pos.device, dtype=batch.pos.dtype))
masked_forward = masked_k_forward_sample(k0, sg, times.lattice, model.diffusion_l)
rows_t6 = []
for g in range(int(batch.num_graphs)):
    mask = masked_forward['mask'][g]
    bias = masked_forward['bias'][g]
    k_t = masked_forward['k_t'][g]
    constrained = mask == 0
    free = mask != 0
    rows_t6.append({
        'graph': g,
        'space_group': int(sg[g].item()),
        'family': family_from_sg(int(sg[g].item())),
        'constrained_fixed_max_abs': safe_float(torch.max(torch.abs(k_t[constrained] - bias[constrained])) if torch.any(constrained) else torch.tensor(0.0, device=k_t.device)),
        'free_std': safe_float(torch.std(k_t[free]) if torch.any(free) else torch.tensor(0.0, device=k_t.device)),
        'target_zero_on_constrained': bool(torch.allclose(masked_forward['target_k'][g][constrained], torch.zeros_like(masked_forward['target_k'][g][constrained]), atol=CONSTRAINED_TOL)) if torch.any(constrained) else True,
        'finite': bool(torch.isfinite(k_t).all()),
        'PASS': bool(torch.isfinite(k_t).all() and (not torch.any(constrained) or torch.max(torch.abs(k_t[constrained] - bias[constrained])) <= CONSTRAINED_TOL)),
    })

df_t6 = display_sorted_frame(rows_t6, ['graph'])


,graph,space_group,family,constrained_fixed_max_abs,free_std,target_zero_on_constrained,finite,PASS
0,0,194,hexagonal_or_trigonal,0.0,0.893204,True,True,True
1,1,160,hexagonal_or_trigonal,0.0,1.630486,True,True,True
2,2,65,orthorhombic,0.0,0.697064,True,True,True
3,3,8,monoclinic,0.0,0.915465,True,True,True


### Test 7: Network Dry Forward Without Changing `CSPVNet`

This feeds a masked `k_t` tensor directly into the unchanged network. The point is only to verify shape / finiteness / basic compatibility, not prediction quality.


In [10]:
prepared = model.prepare_training_batch(batch=batch, t=torch.full((int(batch.num_graphs),), float(T_PROBE), device=batch.pos.device, dtype=batch.pos.dtype))

preds_native = model.score_network(
    t=prepared.times.graph,
    pos=prepared.f_t,
    v=prepared.v_t,
    h=prepared.atomic_numbers,
    l=prepared.l_t,
    node_index=prepared.node_index,
    edge_node_index=prepared.edge_node_index,
)

preds_masked = model.score_network(
    t=prepared.times.graph,
    pos=prepared.f_t,
    v=prepared.v_t,
    h=prepared.atomic_numbers,
    l=masked_forward['k_t'],
    node_index=prepared.node_index,
    edge_node_index=prepared.edge_node_index,
)

rows_t7 = [
    {
        'network_input': 'native_l_t',
        'pred_v_shape': tuple(preds_native['v'].shape),
        'pred_l_shape': tuple(preds_native['l'].shape),
        'pred_v_finite': bool(torch.isfinite(preds_native['v']).all()),
        'pred_l_finite': bool(torch.isfinite(preds_native['l']).all()),
        'pred_l_norm': safe_float(torch.linalg.norm(preds_native['l'])),
        'PASS': bool(preds_native['v'].shape == batch.pos.shape and preds_native['l'].shape[-1] == 6 and torch.isfinite(preds_native['l']).all()),
    },
    {
        'network_input': 'masked_k_t',
        'pred_v_shape': tuple(preds_masked['v'].shape),
        'pred_l_shape': tuple(preds_masked['l'].shape),
        'pred_v_finite': bool(torch.isfinite(preds_masked['v']).all()),
        'pred_l_finite': bool(torch.isfinite(preds_masked['l']).all()),
        'pred_l_norm': safe_float(torch.linalg.norm(preds_masked['l'])),
        'PASS': bool(preds_masked['v'].shape == batch.pos.shape and preds_masked['l'].shape == masked_forward['k_t'].shape and torch.isfinite(preds_masked['l']).all()),
    },
]

df_t7 = display_sorted_frame(rows_t7, ['network_input'])


,network_input,pred_v_shape,pred_l_shape,pred_v_finite,pred_l_finite,pred_l_norm,PASS
0,masked_k_t,"(21, 3)","(4, 6)",True,True,4.611792,True
1,native_l_t,"(21, 3)","(4, 6)",True,True,5.810829,True


### Test 8: One-Step Masked Reverse Dry Run

This is still notebook-local temporary code. The reverse step is applied in the free `z = mask * (k - bias)` subspace and then re-masked before decoding.


In [11]:
masked_reverse = masked_k_reverse_step(
    k_t=masked_forward['k_t'],
    pred_k=preds_masked['l'],
    sg=sg,
    t_lattice=times.lattice,
    dt=DT_REVERSE,
    diffusion_l=model.diffusion_l,
)

rows_t8 = []
for g, node_mask in enumerate(graph_masks(batch)):
    k_prev = masked_reverse['k_prev'][g]
    mask = masked_reverse['mask'][g]
    bias = masked_reverse['bias'][g]
    cell_prev = crystal_family.v2m(k_prev[None])[0]
    frac = batch.pos[node_mask].reshape(-1, 3)
    atomic_numbers = batch.atomic_numbers[node_mask]
    structure_prev = build_structure(cell_prev, frac, atomic_numbers)
    detected_prev = detect_sg(structure_prev, symprec=1e-2)
    rows_t8.append({
        'graph': g,
        'space_group': int(sg[g].item()),
        'family': family_from_sg(int(sg[g].item())),
        'constrained_fixed_max_abs': safe_float(torch.max(torch.abs(k_prev[mask == 0] - bias[mask == 0])) if torch.any(mask == 0) else torch.tensor(0.0, device=k_prev.device)),
        'det_cell_prev': safe_float(torch.det(cell_prev)),
        'finite_k_prev': bool(torch.isfinite(k_prev).all()),
        'finite_cell_prev': bool(torch.isfinite(cell_prev).all()),
        'detected_sg_prev': detected_prev,
        'detected_family_prev': family_from_sg(detected_prev) if detected_prev is not None else None,
        'PASS': bool(torch.isfinite(cell_prev).all() and torch.det(cell_prev) > 0 and (not torch.any(mask == 0) or torch.max(torch.abs(k_prev[mask == 0] - bias[mask == 0])) <= CONSTRAINED_TOL)),
    })

df_t8 = display_sorted_frame(rows_t8, ['graph'])


,graph,space_group,family,constrained_fixed_max_abs,det_cell_prev,finite_k_prev,finite_cell_prev,detected_sg_prev,detected_family_prev,PASS
0,0,194,hexagonal_or_trigonal,0.0,0.577606,True,True,194,hexagonal_or_trigonal,True
1,1,160,hexagonal_or_trigonal,0.0,11.001970,True,True,8,monoclinic,True
2,2,65,orthorhombic,0.0,0.675423,True,True,47,orthorhombic,True
3,3,8,monoclinic,0.0,1.250278,True,True,1,triclinic,True


## Summary

The table below is not a scientific verdict. It is only a compatibility summary for the masked-lattice idea on the current KLDM pipeline.


In [ ]:
summary_rows = [
    {
        'check': 'batch_fields_ready',
        'PASS': bool(df_t0['PASS'].all()),
        'details': 'All required batch fields, including space_group, are present.'
    },
    {
        'check': 'k_roundtrip_stable',
        'PASS': bool(df_t12['PASS_roundtrip'].all()),
        'details': f"max_roundtrip_rel_error={safe_float(df_t12['roundtrip_rel_error'].max())}"
    },
    {
        'check': 'de_so3_geometry_preserved',
        'PASS': bool(df_t12['PASS_rotation_gauge'].all()),
        'details': f"max_sym_distance_max_abs={safe_float(df_t12['sym_distance_max_abs'].max())}"
    },
    {
        'check': 'raw_gt_projection_not_too_distorting',
        'PASS': bool(df_t345.loc[df_t345['cell_source'] == 'raw_gt', 'PASS_distortion_smallish'].all()),
        'details': f"raw_gt_median_distortion={safe_float(df_t345.loc[df_t345['cell_source'] == 'raw_gt', 'distance_distortion_mean_rel'].median())}"
    },
    {
        'check': 'standardized_gt_projection_not_too_distorting',
        'PASS': bool(df_t345.loc[df_t345['cell_source'] != 'raw_gt', 'PASS_distortion_smallish'].all()),
        'details': f"standardized_median_distortion={safe_float(df_t345.loc[df_t345['cell_source'] != 'raw_gt', 'distance_distortion_mean_rel'].median())}"
    },
    {
        'check': 'masked_vp_forward_valid',
        'PASS': bool(df_t6['PASS'].all()),
        'details': f"max_constrained_fixed_max_abs={safe_float(df_t6['constrained_fixed_max_abs'].max())}"
    },
    {
        'check': 'network_accepts_masked_k',
        'PASS': bool(df_t7['PASS'].all()),
        'details': 'Unchanged CSPVNet consumed masked k_t and returned finite 6D lattice outputs.'
    },
    {
        'check': 'one_step_masked_reverse_valid',
        'PASS': bool(df_t8['PASS'].all()),
        'details': f"min_det_cell_prev={safe_float(df_t8['det_cell_prev'].min())}"
    },
]

summary_df = display_sorted_frame(summary_rows, ['check'])

raw_ok = bool(df_t345.loc[df_t345['cell_source'] == 'raw_gt', 'PASS_structure_match'].all()) if len(df_t345.loc[df_t345['cell_source'] == 'raw_gt']) else False
std_ok = bool(df_t345.loc[df_t345['cell_source'] != 'raw_gt', 'PASS_structure_match'].all()) if len(df_t345.loc[df_t345['cell_source'] != 'raw_gt']) else False
if raw_ok:
    print('Notebook verdict: raw GT lattice masking already looks geometry-compatible on this probe batch.')
elif std_ok:
    print('Notebook verdict: standardized GT cells look compatible, but raw GT cells do not; frame/standardization mismatch is the likely culprit.')
else:
    print('Notebook verdict: even standardized GT cells do not pass cleanly; do not trust hard masked-k training yet.')



,check,PASS,details
0,batch_fields_ready,True,"All required batch fields, including space_gro..."
1,de_so3_geometry_preserved,True,max_sym_distance_max_abs=7.152557373046875e-07
2,k_roundtrip_stable,True,max_roundtrip_rel_error=3.2344601663680805e-07
3,masked_vp_forward_valid,True,max_constrained_fixed_max_abs=0.0
4,network_accepts_masked_k,True,Unchanged CSPVNet consumed masked k_t and retu...
5,one_step_masked_reverse_valid,True,min_det_cell_prev=0.5776058435440063
6,projection_not_too_distorting,False,median_distance_distortion_mean_rel=0.06135569...


Notebook verdict: at least one compatibility gate failed; fix data/operator issues before training.
